Problem Statement: Employee attrition increases hiring costs and reduces productivity. By predicting employees who are likely to leave, organizations can take proactive measures to improve retention and workforce stability.

**Task 1 - Data Loading & Exploration**

In [84]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve)

In [85]:
os.makedirs("charts", exist_ok=True)

In [86]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

In [87]:
df.head(10)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2
5,32,No,Travel_Frequently,1005,Research & Development,2,2,Life Sciences,1,8,...,3,80,0,8,2,2,7,7,3,6
6,59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,1,10,...,1,80,3,12,3,2,1,0,0,0
7,30,No,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,1,11,...,2,80,1,1,2,3,1,0,0,0
8,38,No,Travel_Frequently,216,Research & Development,23,3,Life Sciences,1,12,...,2,80,0,10,2,3,9,7,1,8
9,36,No,Travel_Rarely,1299,Research & Development,27,3,Medical,1,13,...,2,80,2,17,3,2,7,7,7,7


In [88]:
row, cols = df.shape

In [89]:
print(row)
print(cols)

1470
35


In [ ]:
print(df["Attrition"].head())

In [ ]:
print(df["Attrition"].value_counts()) #target variable details

In [ ]:
attrition_counts = df["Attrition"].value_counts()
attrition_rate = (attrition_counts['Yes']/row)*100
print(f"{attrition_rate:.2f}%")

In [ ]:
numeric_cols = df.select_dtypes(include=["number"]).columns
categorical_cols = df.select_dtypes(include=["object", "string"]).columns

In [ ]:
print(f"Number of Numeric Columns: {len(numeric_cols)}")
print(f"Number of Categorical Columns: {len(categorical_cols)}")

Is the attrition rate balanced or imbalanced?
The dataset contains 1,470 employee records, of which 237 employees (16.12%) left the company and 1,233 employees (83.88%) stayed. This indicates that the dataset is imbalanced, as the majority of records belong to employees who did not leave. Therefore, relying solely on accuracy may be misleading, and evaluation metrics such as Precision, Recall, F1-score, and ROC-AUC should be used to assess model performance more effectively.

**Task 2 - Data Cleaning & Preprocessing**

In [ ]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0] #no missing values acc to output 

In [ ]:
columns_to_drop = [
    "EmployeeNumber",   
    "EmployeeCount",   
    "Over18",           
    "StandardHours"    
]

df.drop(columns=columns_to_drop, inplace=True) #irrelevant data dropped

In [ ]:
binary_cols = ["Gender", "OverTime"]
binary_mapping = {"Yes": 1, "No": 0, "Male": 1, "Female": 0}

for col in binary_cols:
    df[col] = df[col].map(binary_mapping)

df["Attrition"] = df["Attrition"].map({
    "Yes": 1,
    "No": 0
}) #applied binary encoding to all binary categorical columns

In [ ]:
df[binary_cols + ["Attrition"]].head()

In [ ]:
X = df.drop("Attrition", axis=1)
y = df["Attrition"]

In [ ]:
categorical_cols = X.select_dtypes(include=["object", "string"]).columns

print(categorical_cols)

In [ ]:
X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [ ]:
#feature scaling applied after the train-test split to prevent data leakage

**Task 3 - EDA**

In [ ]:
dept_attrition = df.groupby("Department")["Attrition"].mean() * 100
print("Attrition Rate by Department:", dept_attrition.sort_values(ascending=False)) 

**which department loses the most employees?**  
sales dept loses the most employees

In [ ]:
job_attrition = df.groupby("JobRole")["Attrition"].mean() * 100
print("Attrition Rate by Job Role:", job_attrition.sort_values(ascending=False))

**which roles have the highest exit rate?**  
Sales Representative have the highest exit rate

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Attrition", y="MonthlyIncome", hue="Attrition", palette="Set2", legend=False)
plt.title("Monthly Income Distribution: Leavers vs Stayers")
plt.xlabel("Attrition \n (0 = Stayed, 1 = Left)")
plt.ylabel("Monthly Income")
plt.tight_layout()
plt.savefig("charts/attrition_vs_monthly_income.png")
plt.show()

In [ ]:
median_income = df.groupby("Attrition")["MonthlyIncome"].median()
print(f"Median Income for Employees who Stayed: {median_income[0]:,.2f}")
print(f"Median Income for Employees who Left: {median_income[1]:,.2f}\n") 

**do lower paid employees leave more?**  
median income for employees who left is significantly less than median income for employees who stayed. yes, lower paid employees leave more

In [ ]:
wlb_rates = df.groupby("WorkLifeBalance")["Attrition"].mean() * 100

plt.figure(figsize=(7, 5))
sns.barplot(x=wlb_rates.index, y=wlb_rates.values, hue=wlb_rates.index, palette="Oranges_r", legend=False)
plt.title("Attrition Rate by Work-Life Balance Rating")
plt.xlabel("Work-Life Balance Rating \n (1 = Bad, 4 = Best)")
plt.ylabel("Attrition Rate (%)")
plt.ylim(0, 40)

for i, rate in enumerate(wlb_rates.values):
    plt.text(i, rate + 1, f"{rate:.1f}%", ha="center", weight="bold")

plt.tight_layout()
plt.savefig("charts/attrition_vs_worklife_balance.png")
plt.show()

for rating, rate in wlb_rates.items():
    print(f"Rating {rating}: {rate:.2f}% Attrition")
print("\n")

**is there a visible pattern?**  
There is a strong pattern, but it is not a perfect downward trend because of the small increase at Rating 4. The main takeaway is that very poor work-life balance is a major risk factor for attrition.

In [ ]:
plt.figure(figsize=(12, 5))

sns.countplot(data=df[df["YearsAtCompany"] <= 15], x="YearsAtCompany", hue="Attrition", palette="viridis")
plt.title("Employee Count by Tenure at Company (First 15 Years)")
plt.xlabel("Years at Company")
plt.ylabel("Number of Employees")
plt.legend(title="Attrition", labels=["Stayed (0)", "Left (1)"])
plt.tight_layout()
plt.savefig("charts/attrition_vs_years_at_company.png")
plt.show()

tenure_rates = df.groupby("YearsAtCompany")["Attrition"].mean() * 100
for year in range(6):
    print(f"Year {year} at Company Attrition Rate: {tenure_rates.get(year, 0):.2f}%")

**at what point in tenure do employees leave most?**

there are two ways to interpret this.  
1. Highest number of employees leaving
2. Highest attrition rate

highest number of employees leaving - the largest number of employees leave during their 1st year at the company. years 2–5 also have employees leaving, but the counts are noticeably lower  
highest attrition rate - employees are most likely to leave within their first year of employment (Years 0–1)

**Business insights from EDA**

1. High Risk in the First Year
New hires at Year 0 have a very high exit rate of 36.36%, which stays nearly the same at 34.50% for employees in their first full year (Year 1). After the second year, the rate drops sharply to 21.26% and then stabilizes. This means HR needs to focus heavily on support during the first 12 to 24 months to keep new hires from leaving.

2. Overtime Drives Employees Away
Employees who work Overtime have an attrition rate of 30.53%, compared to only 10.44% for those who do not work extra hours. Routinely giving employees overtime work nearly triples their likelihood of leaving the company. This shows that workload fatigue is a major reason people quit.

3. Low Salaries Equal Higher Turnover
The data shows a clear salary gap: the median monthly income for employees who leave is $3,202, while the median income for those who stay is $5,204. Attrition is heavily concentrated among lower-paid or entry-level staff. These employees are the most likely to change companies for better pay.

4. Sales Reps and Lab Techs Have the Highest Exit Rates
Turnover is not equal across all departments; Sales Representatives have a massive 39.76% attrition rate, followed by Laboratory Technicians at 23.94%. Meanwhile, higher roles like Managers (4.90%) and Research Directors (2.50%) rarely leave. Retention strategies should be targeted directly at these high-risk roles rather than across the whole company.

5. Poor Work-Life Balance Cuts Retention in Half
Employees who rate their Work-Life Balance as a 1 (Poor) have an attrition rate of 31.25%. However, for those who rate it a 2 (Good) or 3 (Excellent), the exit rate drops immediately to 16.86% and 14.22%. Even small improvements to workload distribution can save half of the employees who are currently at risk of quitting.

**Task 4 - Model Building & Comparison**

In [ ]:
current_numeric_cols = X.select_dtypes(include=["number"]).columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()
X_train[current_numeric_cols] = scaler.fit_transform(X_train[current_numeric_cols])
X_test[current_numeric_cols] = scaler.transform(X_test[current_numeric_cols])

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000),
    "Random Forest Classifier": RandomForestClassifier(class_weight="balanced", random_state=42),
    "Gradient Boosting Classifier": GradientBoostingClassifier(random_state=42)
}

comparison_data = []

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_prob)
    
    comparison_data.append({
        "Model Name": name,
        "ROC-AUC Score": round(roc_auc, 4)
    }) #training model and finding auc score

In [ ]:
comparison_table = pd.DataFrame(comparison_data)
print(comparison_table) #printing comparison table

#AUC Score - Logistic Regression > Gradient Boosting Classifier > Random Forest CLassifier 

**Task 5 - Model Evaluation** 

In [ ]:
predictions = {}

for name, model in models.items():
    print(f"\n Performance Evaluation: {name}")

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    predictions[name] = {"y_pred": y_pred, "y_prob": y_prob}

    print(classification_report(y_test, y_pred))

**Identify and clearly state which model performed best and why**  
Logistic Regression had the best ROC-AUC (0.7983), just ahead of Gradient Boosting (0.7940) and well ahead of Random Forest (0.7471). Since only 16% of employees left, accuracy alone is misleading andrecall matters more. Logistic Regression catches 62% of actual leavers versus 21% for Gradient Boosting and just 9% for Random Forest. Logistic Regression is the best model because it's the most useful one for actually flagging at-risk employees.

In [ ]:
best_model_name = comparison_table.loc[comparison_table["ROC-AUC Score"].idxmax(), "Model Name"]
best_model = models[best_model_name]

y_pred = predictions[best_model_name]["y_pred"]
y_prob = predictions[best_model_name]["y_prob"]

print(f"Best Model (by ROC-AUC): {best_model_name}")

In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score ({best_model_name}): {roc_auc:.4f}")  #auc score for the best model only

In [ ]:
print(f"Confusion Matrix ({best_model_name}):")
print(confusion_matrix(y_test, y_pred))
print("-" * 50)

Logistic Regression was picked as the best model above, but its coefficients aren't great for ranking importance once features are one-hot encoded and scaled. Random Forest's feature_importances_ is used here instead, just to rank which factors matter most and it's a common substitution for interpretability, not a change in which model we're actually using. Predictions and metrics above still come from Logistic Regression.

In [ ]:
rf_model = models["Random Forest Classifier"]
importances = rf_model.feature_importances_

feature_importances = pd.Series(importances, index=X.columns).sort_values(ascending=False)

In [ ]:
print("Ranked Top 10 Most Important Features Driving Employee Exit: \n") 
print(feature_importances.head(10))

**Task 6 - Visualization**

In [ ]:
#Chart 1: Bar chart showing attrition rate by Department and Job Role

plt.figure(figsize=(12, 6))

sns.barplot(data=df, x="JobRole", y="Attrition", hue="Department", errorbar=None)
plt.xticks(rotation=45, ha="right")
plt.title("Attrition Rate by Job Role and Department")
plt.xlabel("Job Role")
plt.ylabel("Attrition Rate (Proportion)")
plt.tight_layout()
plt.savefig("charts/chart1_attrition_by_role_dept.png")

plt.show()

In [ ]:
#Chart 2: Box plot comparing Monthly Income of employees who left vs stayed

plt.figure(figsize=(7, 5))

sns.boxplot(data=df, x="Attrition", y="MonthlyIncome", hue="Attrition", palette="Set2", legend=False)
plt.title("Monthly Income Comparison: Stayed (0) vs Left (1)")
plt.xlabel("Attrition Status")
plt.ylabel("Monthly Income ($)")
plt.tight_layout()
plt.savefig("charts/chart2_monthly_income_boxplot.png")

plt.show()

In [ ]:
#Chart 3: Confusion Matrix heatmap for your best model

best_model = models["Logistic Regression"]

y_pred_lr = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=["Predicted Stayed", "Predicted Left"],
            yticklabels=["Actual Stayed", "Actual Left"])
plt.title("Confusion Matrix Heatmap - Logistic Regression")
plt.tight_layout()
plt.savefig("charts/chart3_confusion_matrix.png")

plt.show()

In [ ]:
#Chart 4: Horizontal bar chart of Top 10 Feature Importances from your best model

top_10_features = feature_importances.head(10).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
top_10_features.plot(kind="barh", color="teal")
plt.title("Top 10 Most Important Features Driving Employee Exit")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.tight_layout()
plt.savefig("charts/chart4_feature_importances.png")
plt.show()

In [ ]:
#Chart 5: ROC Curve comparing all 3 models on one graph

plt.figure(figsize=(8, 6))

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.4f})")

plt.plot([0, 1], [0, 1], color="navy", linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity / Recall)")
plt.title("ROC Curve Comparison - All 3 Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("charts/chart5_roc_curve_comparison.png")
plt.show()